# Dynamic Programming

> extension of the theory of Markov Chains to Markov Decision Processes (MDP)

A *MDP* is a stochastic process caractherised by the following quintuplet: $(\mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma)$, where:

- $\mathcal{S}$ is the set of states;

- $\mathcal{A}$ is the set of actions;

- $\mathcal{P}:=\{P^{a}: a\in\mathcal{A}\}$ is the set of transition matrices for each fixed action;

- $\mathcal{R}$ is the set of rewards obtained by applying an action to a state. We denote by $R_{n+1}$ the reward obtained by applying the action $A_{n}$ to state $S_{n}$, at time $n\in\mathbb{N}^{+}$;

- $\gamma\in [0,1)$ is the penalisation factor.

Instaed of dealing with random variable rewards $R_{n+1}$, we usually consider:

- the average reward $\bar{R}_{n+1}(s,a)=\mathbb{E}[R_{n+1} \mid S_{n}=s, A_{n}=a]$, kwnowing both the state and action;

- the average reward $\bar{R}_{n+1}(s)=\mathbb{E}[R_{n+1} \mid S_{n}=s]$, kwnowing only the state.

We call a *policy* $\pi$ the set of markovian decision rules $\{ \mathbb{P}(A_{n}=a \mid S_{n}=s) : n\in \mathbb{N}^{+}, a\in \mathcal{A}, s\in\mathcal{S}\}$.

In our code, we will only consider *stationary policies*, i.e. not depending on time $n\in\mathbb{N}^{+}$.

**What is the goal of Dynamic Programming ?** 

In a MDP, at each time $n\in \mathbb{N}^{+}$, an action $A_{n}=a$ is applied to a state $S_{n}=s$, and the system receives a reward. Usually, the average value of the reward is positive if we want to encourage this action (good for the system), negative if we want to discourage (bad for the system), or zero if it is ininfluential for the system. The goal of Dynamical Programming is learning which choise of actions maximises the reward.

## Value Function

> The average future reward.

Considering $\mathcal{S}$ finite, there are two ways of computing the value function $v_{\pi}$:

1) exactly by inverting a matrix (affordable method if the state space is small);

2) as the limit of a converging sequence.

### 1) Exact method:

 $\quad v_{\pi} = \left(I - \gamma P_{\pi}\right)^{-1} \bar{R}_{\pi}$,

where $I$ is the squared matrix of dimension $|\mathcal{S}|\times |\mathcal{S}|$, $P_{\pi}$ is the transition matrix of the chain of states, and $\bar{R}_{\pi}$ is the vector of average rewards for each state.


In [1]:
import numpy as np


In [2]:
def R_weighted(R, pi):

    '''
    Given the rewards and the policy, it computes the average reward
    
    R[s, a] : R_{1}(s, a) rewards

    pi[s, a] : policy law pi(a | s)

    '''

    if R.shape[0] != pi.shape[0] or R.shape[1] != pi.shape[1]:
        raise ValueError("matrices R[s,a] and pi must have same size")
    
    return np.sum(R * pi, axis=1)

In [3]:
def P_wighted(P_act, pi):

    '''
    Given a list of stochastic matrices for each action: P_act[s,s',a]=P_{s,s'}^{a}

    and a policy pi[s, a]=pi(a | s),

    return the weighted stochastic matrix P_pi[s,s']
    '''

    n_states, n_next_states, n_actions = P_act.shape

    if pi.shape[0] != n_states or pi.shape[1] != n_actions:
            raise ValueError("matrices P_{s,s'}^a and pi must have compatible sizes")


    P_pi = np.sum(P_act * pi[:, None, :], axis=2)            #sum on a

    return P_pi

In [4]:

def value_function_exact(gamma, P_pi, R_pi):

    '''
    It computes the value function by inverting the matrix (I - gamma* P)
    
    gamma: penalisation factor in [0, 1)

    P_{\pi}[s, s']: transition matrix of states chain

    R_{\pi}[s]: vector of average rewards for each state
    '''

    P_pi = np.asarray(P_pi, dtype=float)
    R_pi = np.asarray(R_pi, dtype=float)

    #size checks
    if P_pi.shape[0] != P_pi.shape[1]:
        raise ValueError("P_pi must be a square transition matrix.")
    if R_pi.shape[0] != P_pi.shape[0]:
        raise ValueError("R_pi must contain one reward per state.")
    if not 0 <= gamma < 1:
        raise ValueError("gamma must satisfy 0 <= gamma < 1 .")

    #return the value function
    id = np.eye(P_pi.shape[0])

    inv = np.linalg.inv(id - gamma*P_pi)

    return np.dot(inv, R_pi)

    

Example from the course. The particularity here is that we have three absorbing states, for which the rewards are set to zero by convention.

In [5]:
P_pi = np.array([                                       #stochastic matrix of states
    [0.5, 0.5, 0, 0, 0, 0, 0],
    [0.3, 0, 0.7, 0, 0, 0, 0],
    [0, 0, 0.5, 0.5, 0, 0, 0],
    [0, 0, 0, 0.1, 0, 0.7, 0],
    [0, 0, 0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 0, 0, 1]
])

R_pi = np.array([0.5, -0.7, -5.5, 89, 0, 0, 0]).T       #average rewards vector

gamma_ex = 0.8

value_function_exact(gamma_ex, P_pi, R_pi)


array([25.02587992, 36.28881988, 55.32608696, 96.73913043,  0.        ,
        0.        ,  0.        ])

### 2) Iterative method:

$\begin{cases}
v_{0} = v_{0}' \\
v_{n+1} = L_{\pi} v_{n}
\end{cases}$,


where $L_{\pi}v(s) = \bar{R}_{\pi}(s) + \gamma \mathbb{E}_{\pi}\left[v(S_{1}) | S_{0}=s\right]$, for each $v \in \mathbb{R}^{|\mathcal{S}|}$.

If we develop explicitely, we obtain $L_{\pi}v(s) = \sum_{a\in \mathcal{A}} \left( \bar{R}_{1}(s, a) + \gamma \left( \sum_{s'\in \mathcal{S}} P_{s,s'}^{a} v(s') \right)\right) \pi(a \mid s)$.

In [18]:

def value_function_iterative(gamma, P, pi, v0, R=None, R_bar=None, res=1, eps=1e-4, max_iter=1000):

    '''
    It computes the value function by iterative method, given the policy
    
    gamma: penalisation factor in [0, 1)

    P[s, s', a]: list of action transition matrices

    R[s, a]: rewards with shape (states, actions)

    R_bar[s]: vector of average rewards R_{pi}(s)
                   
            :if R_bar is provided, it's okay
            :if R_bar is not provided, it is computed from R

    pi[s, a]: policy law \pi(a | s)

    v0: initial value vector

    res: residual between new and old vector

    eps: tolerance criteria

    iter: number of iterations

    '''


    P = np.asarray(P, dtype=float)
    pi = np.asarray(pi, dtype=float)

    
    n_states, n_next_states, n_actions = P.shape
    
    if n_states != n_next_states:
        raise ValueError("P must be a squared matrix, for each fixed action.")
    if pi.shape[0] != n_states or pi.shape[1] != n_actions:
        raise ValueError("pi must have shape (states, actions).")
    if len(v0) != n_states:
        raise ValueError("v0 must have one value per state.")
    if not 0 <= gamma < 1:
        raise ValueError("gamma must satisfy 0 <= gamma < 1.")
    if eps <= 0:
        raise ValueError("tolerance eps must be positive.")
    if max_iter <= 0:
        raise ValueError("number of iterations iter must be positive.")



    #compute expected rewards only if R_bar is not provided and size checks
    if R_bar is None:                                                    #R_bar not provided

        if R is None:                                                    #provide at least one between R_bar or R
            raise ValueError("Provide either R or R_bar.")
        elif R.shape[0] != n_states or R.shape[1] != n_actions:
            raise ValueError("R must have shape (states, actions)")

        R = np.asarray(R, dtype=float)   
        R_bar = average_reward(R, pi)                                    #compute the average reward

    else:                                                                #R_bar provided

        R_bar = np.asarray(R_bar, dtype=float)                               
        if R_bar.shape[0] != n_states:
            raise ValueError("R_bar must have shape (n_states,).")


    #ALGORITHM
    v_new = np.asarray(v0, dtype=float)

    n = 0
    
    while res >= eps and n < max_iter:

        v = v_new.copy()                                                #associate the updated vector

        inner_term = np.sum(P * v[None, :, None], axis=1)               #sum over s'

        v_new = R_bar + (gamma * np.sum(inner_term * pi, axis=1) )      #update vector v, with sum over actions on second term

        res = np.max(np.abs(v_new - v))                                 #compute residual
        
        n += 1

    return v_new



Example from the course.

Notice that, for absorbing terminal states $i$:

- the policy $\pi(i \mid a)$ can be arbitrarly chosen, provided $\sum_{i} \pi(i \mid a) = 1$;
- the initial vector $v_{0}$ must have entries $v_{0}[i] = 0$. Indeed, the maximum reward in an absorbing state is always set to $0$ by convention.

In [7]:
#R_ex already provided

#policy
pi_ex = np.zeros((7, 2))         

pi_ex[0, 0] = 1
pi_ex[1, 1] = 1
pi_ex[2, 1] = 1
pi_ex[3, 0] = 1

pi_ex[4, 0] = pi_ex[5, 0] = pi_ex[6, 0] = 1    #absorbing states


#transition matrices for each action
P_0ex = np.array([[0.5, 0.5, 0, 0, 0, 0, 0],
                [0, 0.6, 0, 0, 0.4, 0, 0],
                [0, 0.4, 0.6, 0, 0, 0, 0],
                [0, 0, 0, 0.1, 0, 0.9, 0],
                [0, 0, 0, 0, 1, 0, 0],
                [0, 0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 0, 1]])

P_1ex = np.array([[0.5, 0, 0.5, 0, 0, 0, 0],
                [0.3, 0, 0.7, 0, 0, 0, 0],
                [0, 0, 0.5, 0.5, 0, 0, 0],
                [0, 0, 0, 0, 0, 0, 1],
                [0, 0, 0, 0, 1, 0, 0],
                [0, 0, 0, 0, 0, 1, 0],
                [0, 0, 0, 0, 0, 0, 1]])

P_ex = np.stack([P_0ex, P_1ex], axis=2)


#compute the value function via the iterative method

v0_ex = [21, 40, 6, 15, 0, 0, 0]  #random initial vector, with entries for absorbing states set to 0

value_function_iterative(gamma_ex, P_ex, pi_ex, v0_ex, R_bar=R_pi)


array([25.02579055, 36.28878216, 55.32608695, 96.73913043,  0.        ,
        0.        ,  0.        ])

## Optimal Value Function and Optimal Policy

> The maximal average future reward and the actions that maximize it.

There are at least two algorithms computing the optimal value function and therefore the optimal policy:

1) **Algorithm of values iterations**: the value function is updated at each iteration, through an application of the Bellmann Operator.

Pros: low computational cost

Cons: approximation of the optimal value function

2) **Algorithm of policy iteration**: the policy is updated at each iteration, through the exact formula.

Pros: high computational cost

Cons: returns the exact optimal value function, after a finite number of iterations

#### Algorithm of values iterations

The maximal average future reward $v_{\pi}^{*}$, or maximal value function, is the fixed point of the *Bellmann Equation* $L^{*}_{\pi} v = v$, where the *Bellmann operator* is the following:

$L^{*}_{\pi}v(s)= \max\limits_{a\in\mathcal{A}} L_{\pi}v(s) = \max\limits_{a\in\mathcal{A}} \left( \bar{R}_{\pi}(s, a) + \gamma \mathbb{E}_{\pi}\left[v(S_{1}) | S_{0}=s, A_{0}=a \right] \right) =  \max\limits_{a\in\mathcal{A}}   \left( \bar{R}_{1}(s, a) + \gamma \left( \sum_{s'\in \mathcal{S}} P_{s,s'}^{a} v(s') \right)\right) $,

for each $v \in \mathbb{R}^{|\mathcal{S}|}$.

Furthermore, the action $a^{*}$ that maximizes this operator is called the *optimal action*.

The following code come from value_function_iterative, modified where necessary.

In [8]:
def optimal_value_function_iterative(gamma, P, R, v0, res=1, eps=1e-4, max_iter=1000):

    '''
    It computes the value function by iterative method, given the policy
    
    gamma: penalisation factor in [0, 1)

    P[s,s',a]: list of action transition matrices

    R[s,a]: rewards with shape (states, actions)

    pi[s, a]: policy law \pi(a | s)

    v0: initial value vector

    res: residual between new and old vector

    eps: tolerance criteria

    iter: number of iterations

    RETURN: value function v, optimal actions a*

    '''


    P = np.asarray(P, dtype=float)
    R = np.asarray(R, dtype=float)

    
    n_states, n_next_states, n_actions = P.shape
    
    if n_states != n_next_states:
        raise ValueError("P must be a squared matrix, for each fixed action.")
    if R.shape[0] != n_states or R.shape[1] != n_actions:
        raise ValueError("R must have shape (states, actions).")
    if len(v0) != n_states:
        raise ValueError("v0 must have one value per state.")
    if not 0 <= gamma < 1:
        raise ValueError("gamma must satisfy 0 <= gamma < 1.")
    if eps <= 0:
        raise ValueError("tolerance eps must be positive.")
    if max_iter <= 0:
        raise ValueError("number of iterations iter must be positive.")



    #ALGORITHM
    v_new = np.asarray(v0, dtype=float)                                 #initial association

    n = 0                                                               #iterations
    
    while res >= eps and n < max_iter:

        v = v_new.copy()                                                #associate the updated vector

        inner_term = np.sum(P * v[None, :, None], axis=1)               #sum over s', returns array [s, a]

        v_new = np.max(R + gamma * inner_term, axis=1)                  #update vector v, maximizes over actions on axis=1

        action_opt = np.argmax(R + gamma * inner_term, axis=1)          #optimal actions

        res = np.max(np.abs(v_new - v))                                 #compute residual
        
        n += 1

    return v_new, action_opt


Example from the course. We use the same data of the previous examples.

We find that the optimal actions are $a^{*}=[1, 1, 1, 0, 0, 0, 0]$. This means that, to maximize the average future reward, in states $1, 2, 3$ the best action  is $1$, otherwise it is $0$. In other words:

$\pi^{*}(1 \mid s)$  if $s\in\{1,2,3\}$

$\pi^{*}(0 \mid s)$  if $s\in\{4,5,6,7\}$

is the optimal policy.

In [9]:
#gamma_ex, P_ex, v0_ex already given

#matrix of average rewards knowing both states s and actions a

R_total_ex = np.array([[0.5, -0.5],
                       [-3.4, -0.7],
                       [-0.2, -5.5],
                       [89, -1000],
                       [0, 0],
                       [0, 0],
                       [0, 0]])


optimal_value_function_iterative(gamma_ex, P_ex, R_total_ex, v0_ex)


(array([36.05066537, 38.93474441, 55.32608366, 96.73913043,  0.        ,
         0.        ,  0.        ]),
 array([1, 1, 1, 0, 0, 0, 0]))

### Algorithm of policy iterations

This algorithm mixes up the two previous ones. It is composed of two phases, repeated alternating:

**Phase 1**: policy updated, using the exact formula $v_{\pi_{k}} = (I - \gamma P_{\pi})^{-1} \bar{R}_{\pi_{k}}$.

**Phase 2**: compute optimal action $a^{*}$, by solving the Bellmann Equation as done in the Algorithm of Values Iterations.

**Phase 3**: update optimal policy $\pi_{k}$.

In [10]:

def policy_iteration(gamma, P, R, pi0, v0, max_iter=1000):

    '''
    Computes the optimal value function and optimal actions using policy iteration.

    gamma : penalisation factor in [0, 1).

    P[s,s',a] : transition probability matrices, for each action.

    R[s, a] : average rewards.

    pi0[s, a] : initial policy pi(a | s).

    v0 : initial value function.

    max_iter : maximum number of iterations

    RETURN: value function v, optimal actions a*, optimal policy \pi
    '''

    P = np.asarray(P, dtype=float)
    R = np.asarray(R, dtype=float)


    pi = np.asarray(pi0, dtype=float).copy()         #initial law
    v = np.asarray(v0, dtype=float).copy()           #initial value function

    n_states, n_next_states, n_actions = P.shape

    # Check dimensions
    if n_states != n_next_states:
        raise ValueError("P must be square for each action.")
    if R.shape[0] != n_states or R.shape[1] != n_actions:
        raise ValueError("R must have shape (n_states, n_actions).")
    if pi.shape[0] != n_states or pi.shape[1] != n_actions:
        raise ValueError("pi0 must have shape (n_states, n_actions).")
    if v.shape[0] != n_states:
        raise ValueError("v0 must have shape n_states.")
    if not 0 <= gamma < 1:
        raise ValueError("gamma must satisfy 0 <= gamma < 1.")
    if max_iter <= 0:
        raise ValueError("max_iter must be positive.")
    if np.any(pi < 0) or not np.allclose(pi.sum(axis=1), 1):
        raise ValueError("Each row of pi0 must be a probability distribution.")



    #ALGORITHM OF POLICY ITERATION
    for _ in range(max_iter):

        #PHASE 1: UPDATE VALUE FUNCTION BY EXACT METHOD
        P_pi = P_wighted(P, pi)                  #P_pi

        R_pi = R_weighted(R, pi)             #R_pi

        v = value_function_exact(gamma, P_pi, R_pi)

        #PHASE 2: COMPUTE OPTIMAL ACTION
        inner_term = np.sum(P * v[None, :, None], axis=1)          #sum on s'

        action_opt = np.argmax( R + gamma * inner_term, axis=1)    #optimal action

        #PHASE 3: UPDATE OPTIMAL POLICY
        pi_new = np.zeros_like(pi)

        pi_new[np.arange(n_states), action_opt] = 1.0                  #set up the optimal policy

        #If the policy has not changed, i.e. it is optimal, break the loop
        if np.array_equal(pi_new, pi):
            return v, action_opt, pi_new

        #set up new policy if the loop continues
        pi = pi_new



    return v, action_opt, pi


Example from the course. Returns in order: optimal value function, optimal actions, optimal policy.

In [11]:
policy_iteration(gamma_ex, P_ex, R_total_ex, pi_ex, v0_ex)

(array([36.05072464, 38.93478261, 55.32608696, 96.73913043,  0.        ,
         0.        ,  0.        ]),
 array([1, 1, 1, 0, 0, 0, 0]),
 array([[0., 1.],
        [0., 1.],
        [0., 1.],
        [1., 0.],
        [1., 0.],
        [1., 0.],
        [1., 0.]]))

On our example from the course, we happily conclude that:

- both the exact and iterative method compute the same value function.

- both the value iterations algorithm and the policy iterations algorithm compute the same optimal value function and optimal policy.

#### **Second toy example** from the course.

State space of dimension $2$ and actions space of dimension $3$. Observations:

- We try to change the initial policy for the algorithm of policy iterations, and we always get the same optimal policy: $\pi(1 \mid 0) = \pi(0 \mid 1) = 1$. 

- We try to change the policy, and we get different value functions (as expected, because the computation of $v_{\pi}$ is directly dependent of the single policy $\pi$ in input).

In [12]:
Pw = np.array([[1, 0], [1,0]])
Ps = np.array([[0.3, 0.7], [0.5,0.5]])
Pr = np.array([[1,0], [1,0]])

P_actions_2 = np.stack((Pw, Ps, Pr), axis=2)

R_2 =np.array([[1, 5, 0], [1, 1, 0]])

v0_2 = [6, 10]

gamma_2 = 0.9

#------------------------


pi_2 = np.zeros((2, 3))

pi_2[0, 1] = 1

pi_2[1, 0] = pi_2[1, 2] = 0.5

policy_iteration(gamma_2, P_actions_2, R_2, pi_2, v0_2)


(array([34.5398773 , 32.08588957]),
 array([1, 0]),
 array([[0., 1., 0.],
        [1., 0., 0.]]))

In [13]:
value_function_exact(gamma_2, P_wighted(P_actions_2, pi_2), R_weighted(R_2, pi_2))

array([32.60736196, 29.84662577])

In [14]:
pi_2 = np.zeros((2, 3))

pi_2[0, 1] = pi_2[1, 1] = 1

policy_iteration(gamma_2, P_actions_2, R_2, pi_2, v0_2)


(array([34.5398773 , 32.08588957]),
 array([1, 0]),
 array([[0., 1., 0.],
        [1., 0., 0.]]))

In [15]:
value_function_exact(gamma_2, P_wighted(P_actions_2, pi_2), R_weighted(R_2, pi_2))

array([28.6440678 , 25.25423729])

In [16]:
pi_2 = np.zeros((2, 3))

pi_2[0, 1] = pi_2[1, 2] = 1

policy_iteration(gamma_2, P_actions_2, R_2, pi_2, v0_2)


(array([34.5398773 , 32.08588957]),
 array([1, 0]),
 array([[0., 1., 0.],
        [1., 0., 0.]]))

In [17]:
value_function_exact(gamma_2, P_wighted(P_actions_2, pi_2), R_weighted(R_2, pi_2))

array([30.67484663, 27.60736196])